# Swap the model into the form

The form reads two files, `models/classic.joblib` and `models/mlp.joblib`, and a small `models/chosen.json`. The radio switches which file is scored. The version line comes from `chosen.json`, which this notebook writes after registering one model in MLflow.

Register under the name `fresh-and-good-churn`. The starter registers the MLP so the form can show a version. If the memo says the forest, change `chosen` and run the cell again. A second register creates version 2.

Fill in `why_this_model.md` before you leave the block. Accuracy, explainability, and cost.


In [ ]:
import os
import sys
from pathlib import Path

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    for folder in (candidate, candidate / "04-deep-learning"):
        marker = folder / "lib" / "customers.py"
        if marker.is_file() and (folder / "app.py").is_file():
            root = folder
            break
    if root is not None:
        break
if root is None:
    raise FileNotFoundError("Start Jupyter from the repo folder or from 04-deep-learning.")
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("module", ROOT)


In [ ]:
import json

from customers import compare_models, load_customers, save_bundle, split_frame
from tracking import REGISTERED_NAME, register_pipeline

customers = load_customers()
train, validation, test = split_frame(customers)
result = compare_models(train, validation, (32,))
print("forest validation ROC-AUC", round(result["classic_validation_roc_auc"], 3))
print("MLP validation ROC-AUC", round(result["mlp_validation_roc_auc"], 3))
print("test rows left unused", len(test))

models = ROOT / "models"
models.mkdir(parents=True, exist_ok=True)
save_bundle(models / "classic.joblib", result["classic"], "classic", "RandomForestClassifier")
save_bundle(models / "mlp.joblib", result["mlp"], "mlp", "MLPClassifier")


## Register one model

Exercise. Set `chosen` to `"classic"` or `"mlp"`. Then complete `why_this_model.md`.


In [ ]:
chosen = "mlp"  # Exercise. "classic" or "mlp".

version = register_pipeline(ROOT, result[chosen], chosen)
(models / "chosen.json").write_text(
    json.dumps(
        {
            "model": chosen,
            "version": version,
            "registered_name": REGISTERED_NAME,
        }
    )
)
print("registered", chosen, "version", version, "name", REGISTERED_NAME)


Open the form from this folder:

```sh
uv run python -m streamlit run app.py
```

The radio should name the model. The MLP line should include the MLflow version you just printed, until you register the other one.
